In [ ]:
import math

class StatisticsFramework:
  ''' Calculates some statistics / fits a regression line to the data.

  - mean, variance, standard_deviation and standard_error methods operate on a list of input numerical data.
  - The display_basic_statistics method shows you how to use them properly (and the correct units)

  - logarithmic fit returns the coefficients that best fit y = a + b log(x)
  - get_logarithmic_PMCC returns the PMCC for this model
  - get_logarithmic_r2 returns the coefficient of determination (the PMCC squared)

  - get_logarithmic_residuals returns a list of residuals (actual - predicted) for the model.
  It takes in x_data, y_data, a, b where a and b are the coefficients.
  It would be useful to plot this against log(x) using a scatter graph - since it visualises the goodness-of-fit of our model.
  We would expect randomly distributed data points about 0. 

  - If we can't do this plot, then get_RMSE takes in the list of residuals and calculates their variation about 0. 
  This is another decent (but not as good) way of showing the goodness-of-fit.
  - See display_logarithmic_fit_statistics for an example of use.

  The whole motivation to evaluate the goodness-of-fit of a logarithmic model is that it will show our trees are operating in-line with theory.
  (We expect insert and search to be O(logN) time algorithms) for all trees.

  - (IMPORTANT) Ideally, the standard errors for each sample should be plotted as vertical error bars on our graphs.
  They show the region in which the sample means are likely to fall into. These work (approximately) like standard deviations -
  we expect ~68% of sample means to fall within 1 std-err of our data point. ~95% within 2 std-errors, and 99.7% within 3. 
  The error bars will give an evaluation of how confident we are in our measurements. 

  - Also, I've kept the methods pretty modular, and so there are some unnecessarily repeated calculations.
  But I've kept it like this because (I'm hoping) it makes the class easier to use.
  Talk to me if you're stuck. 
  '''
        
  def mean(self, data):
    return sum(data) / len(data)
  
  def variance(self, data):
    ''' Calculates the variance in a list of numerical data.
    '''
    n = len(data)
    mean = self.mean(data)
    Sxx = self.__sum_squared_deviation(data, mean)
    return Sxx / (n-1) # this is the sample variance
  
  def standard_deviation(self, data):
    '''Calculates the standard deviation in a list of numerical data.
    '''
    return math.sqrt(self.variance(data))
    
  def standard_error(self, data):
    '''Returns the standard error of the mean for a list of numerical data.
    Equal to, the sample standard deviation / sqrt(n), where n is the number of elements in the list.
    The standard error estimates the standard deviation of the of distribution of sample means. That is, an estimate of how the sample mean varies over repeated samples. 
    '''
    n = len(data)
    return self.standard_deviation(data) / math.sqrt(n)
    
  def logarithmic_fit(self, x_data, y_data):
    '''Return the coefficients for the logarithmic regression of y_data on x_data.
    That is, the values (a,b) such that y = a + b * ln(x) best fits the data.
    '''

    log_x_data = self.__natural_log_data(x_data)
    (a,b) = self.__linear_fit(log_x_data, y_data)

    return (a, b)
  
  def get_logarithmic_PMCC(self, x_data, y_data):
    '''Returns the PMCC of the regression line of y_data on log(x_data).
    '''
    log_x_data = self.__natural_log_data(x_data)
    return self.__get_linear_PMCC(log_x_data, y_data)
  
  def get_logarithmic_r2(self, x_data, y_data):
    ''' Returns r-squared (the PMCC squared). 
    This is a value in the interval [0,1] that tells us the proportion of the variance in the data that the fitted model accurately captures.
    '''
    return self.get_logarithmic_PMCC(x_data, y_data) ** 2
  
  def get_logarithmic_residuals(self, x_data, y_data, a, b):
    '''Return the residuals (actual - predicted) values for the logarithmic fit as a list
    Parameters 'a' and 'b' are the values a, b in the model y = a + b * log(x)
    '''
    n = len(x_data)

    f = lambda x: a + b * math.log(x)

    return [y_data[i] - f(x_data[i]) for i in range(n)]
  
  def get_RMSE(self, residuals):
    '''Returns the RMSE of the model for the fitted data. This is the Root of the Mean of the Squared Errors.
    It is a measure of the spread (standard deviation) of the errors about 0 
    Given by: (the sum of the squares of the residuals / n)^1/2
    '''
    n = len(residuals)
    return math.sqrt(sum([r**2 for r in residuals]) / n)
  
  def display_basic_statistics(self, data):
    '''Calculates and prints the following statistics about a list of numeric data:
    sample mean, sample variance, sample standard deviation and sample standard error
    '''

    sample_mean = self.mean(data)
    sample_variance = self.variance(data)
    sample_std_dev = self.standard_deviation(data)
    sample_std_err = self.standard_error(data)

    # Note: I'm not sure about the units of measurement we are using for time. I have assumed seconds for now.
    print("The basic statistics are: ")
    print(f"The sample mean is: {sample_mean} seconds")
    print(f"The sample variance is: {sample_variance} seconds^2")
    print(f"The sample standard deviation is: {sample_std_dev} seconds")
    print(f"The sample standard error is: {sample_std_err} seconds")
    print(f"\n")

  def display_logarithmic_fit_statistics(self, x_data, y_data):
    '''Fits a logarithmic model, y = a + b * log(x), to y_data and x_data. 
    And gives the following statistics: 
    coefficents of the line of best fit, the PMCC for that line, r^2 (also known as the coefficient of determination, and the RMSE)
    Note, the class also has a method to provide the residuals as list; but this data needs to be plotted.
    '''

    a, b = self.logarithmic_fit(x_data, y_data)
    pmcc = self.get_logarithmic_PMCC(x_data, y_data)
    r2 = self.get_logarithmic_r2(x_data, y_data)
    residuals = self.get_logarithmic_residuals(x_data, y_data, a, b)
    rmse = self.get_RMSE(residuals)

    # Note: I'm not sure about the units of measurement we are using for time. I have assumed seconds for now.
    print("The regression statistics are: ")
    print(f"The model of best fit is: y = {a} + {b}*log(x)")
    print(f"The PMCC is: {pmcc}")
    print(f"The coefficient of determination (r^2) is: {r2}")
    print(f"The RMSE is: {rmse} seconds")
    print(f"\n")

  def __natural_log_data(self, data):
    '''Return a list of the natural logarithms of each value.
    '''
    if any([x <= 0 for x in data]):
      raise ValueError("You cannot log a non-positive number - check logic.")
    return [math.log(i) for i in data]
  
  def __linear_fit(self, x_data, y_data):
    '''Return the coefficients for the linear regression of y_data on x_data.
    Being, (a, b) where y = a + b * x best fits the data.
    Using the formulae: b = Sxy / Sxx - the ratio of covariance and variance
    And, a = y_mean - b * x_mean - the y-intercept of a line with gradient b passing through (x_mean, y_mean)
    '''
    x_mean, y_mean, Sxx, _, Sxy = self.__get_summary_statistics(x_data, y_data)

    b = Sxy / Sxx # Equivalent to the covariance / variance
    a = y_mean - b * x_mean # Line of best fit passes through (x_mean, y_mean)

    return (a, b)
  
  def __get_linear_PMCC(self, x_data, y_data):
    '''Return the product moment correlation coefficient of the linear relationship between y_data and x_data
    Calculation is equivalent to co-variance of x_data and y_data / (standard deviation in x_data * standard deviation in y_data)
    '''

    _, _, Sxx, Syy, Sxy = self.__get_summary_statistics(x_data, y_data)
    return Sxy / math.sqrt(Sxx * Syy)
  
  def __get_summary_statistics(self, x_data, y_data):
    '''Returns some summative statistics about the data used frequently in regression calculations.
    '''

    x_mean = self.mean(x_data)
    y_mean = self.mean(y_data)

    Sxx = self.__sum_squared_deviation(x_data, x_mean)
    Syy = self.__sum_squared_deviation(y_data, y_mean)
    Sxy = self.__sum_co_deviations(x_data, y_data, x_mean, y_mean)

    return (x_mean, y_mean, Sxx, Syy, Sxy)
 
  def __sum_co_deviations(self, x_data, y_data, x_mean, y_mean):
    '''Returns S_xy (a summary statistic for n * covariance) between two lists of numerical data.
    '''
    if len(x_data) != len(y_data):
      raise ValueError("x_data and y_data points are mismatched")
    n = len(x_data)

    return sum([(x_data[i] - x_mean) * (y_data[i] - y_mean) for i in range(n)])
  
  def __sum_squared_deviation(self, x_data, x_mean):
    ''' Returns S_xx (a summary statistic for n * variance) in a list of numerical data.
    '''
    n = len(x_data)

    return sum([(x_data[i] - x_mean)**2 for i in range(n)])

In [ ]:
'''This is a basic demo / test of the stats framework. 
'''
import random

sf = StatisticsFramework()
a, b = 37, 42 # Random test numbers
max_random_offset = 2
x_data = [i for i in range(1, 10)]
f = lambda x: a + b * math.log(x)

y_data = [f(x) for x in x_data]
# Generate y values that closely follow y = a + b log x relationship with some random offset
y_random_offset_data = [y + random.uniform(-max_random_offset, max_random_offset) for y in y_data]

sf.display_basic_statistics(x_data)

sf.display_logarithmic_fit_statistics(x_data, y_data)
sf.display_logarithmic_fit_statistics(x_data, y_random_offset_data)

The basic statistics are: 
The sample mean is: 5.0 seconds
The sample variance is: 7.5 seconds^2
The sample standard deviation is: 2.7386127875258306 seconds
The sample standard error is: 0.9128709291752769 seconds


The regression statistics are: 
The model of best fit is: y = 37.0 + 42.0*log(x)
The PMCC is: 1.0
The coefficient of determination (r^2) is: 1.0
The RMSE is: 0.0 seconds


The regression statistics are: 
The model of best fit is: y = 37.35355740319539 + 41.41140911411013*log(x)
The PMCC is: 0.999157475911227
The coefficient of determination (r^2) is: 0.9983156616692943
The RMSE is: 1.1534840501597083 seconds


